# Day 028 — Exercise 5: ai_analyze_sheet

**What you'll build:** `ai_analyze_sheet(path, question, model='llama3.2') -> str` — reads rows from an XLSX file, converts them to JSON for the LLM context, and returns an AI-generated answer to the given question.

**Why it matters:** This is the AI layer of today's pipeline — the same pattern as `ai_generate_section` (Day 26): system prompt establishes persona, user prompt provides data and question, function returns content string. The `rows[:20]` cap keeps the data within the LLM's context window even for large sheets.

In [ ]:
import json
import ollama
from openpyxl import load_workbook, Workbook

## Provided: read_sheet_rows

In [ ]:
def read_sheet_rows(path: str, sheet_name: str | None = None) -> list[dict]:
    wb = load_workbook(path)
    ws = wb[sheet_name] if sheet_name else wb.active
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return []
    headers = [str(h) for h in rows[0]]
    return [dict(zip(headers, row)) for row in rows[1:]]

## Your Implementation

In [ ]:
def ai_analyze_sheet(
    path: str,
    question: str,
    model: str = 'llama3.2',
) -> str:
    """
    Read an XLSX file and ask an LLM a question about its data.

    Args:
        path:     Path to the .xlsx file.
        question: Question to ask about the data.
        model:    Ollama model name.

    Returns:
        AI-generated answer string.
    """
    # TODO: rows = read_sheet_rows(path)
    # TODO: data_str = json.dumps(rows[:20], indent=2)
    # TODO: system: 'data analyst; answer concisely'
    # TODO: user: f'Data:\n{data_str[:3000]}\n\nQuestion: {question}'
    # TODO: return response['message']['content']
    pass

## Check Your Work

In [ ]:
import tempfile, os


def _run_checks():
    total = 5
    passed = 0

    tmp = tempfile.mktemp(suffix='.xlsx')
    try:
        # Setup: create XLSX with sample data
        wb = Workbook()
        ws = wb.active
        ws.title = 'Sales'
        ws.append(['Product', 'Q1', 'Q2', 'Q3'])
        ws.append(['Alpha', 1200, 1350, 980])
        ws.append(['Beta',   870,  920, 1100])
        ws.append(['Gamma', 2100, 1980, 2200])
        wb.save(tmp)

        # Check 1: defined
        try:
            assert 'ai_analyze_sheet' in globals()
            passed += 1; print('\u2705 Check 1: ai_analyze_sheet defined')
        except Exception as e:
            print(f'\u274c Check 1: {e}')
            print(f'\nScore: {passed}/{total}')
            return

        result = None

        # Check 2: returns a string
        try:
            result = ai_analyze_sheet(
                tmp, 'Which product has the highest Q1 sales?'
            )
            assert isinstance(result, str), \
                f'expected str, got {type(result)}'
            passed += 1; print('\u2705 Check 2: returns a string')
        except Exception as e:
            print(f'\u274c Check 2: {e}')

        # Check 3: result is non-empty
        try:
            assert result is not None
            assert len(result.strip()) > 10, \
                f'response too short ({len(result)} chars): {result!r}'
            passed += 1; print(f'\u2705 Check 3: response is {len(result)} chars')
        except Exception as e:
            print(f'\u274c Check 3: {e}')

        # Check 4: different question gives a response
        try:
            result2 = ai_analyze_sheet(
                tmp, 'How many products are in the data?'
            )
            assert isinstance(result2, str) and len(result2) > 5
            passed += 1; print('\u2705 Check 4: different question also works')
        except Exception as e:
            print(f'\u274c Check 4: {e}')

        # Check 5: result references relevant data (product name present)
        try:
            assert result is not None
            # Gamma has highest Q1 (2100) — LLM should mention it
            assert 'Gamma' in result or 'gamma' in result.lower(), \
                f'expected Gamma mentioned in Q1-highest answer: {result[:200]}'
            passed += 1; print('\u2705 Check 5: answer references the correct product')
        except Exception as e:
            print(f'\u274c Check 5: {e}')

    finally:
        try:
            os.unlink(tmp)
        except Exception:
            pass

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def ai_analyze_sheet(path: str, question: str, model: str = "llama3.2") -> str:
    rows = read_sheet_rows(path)
    data_str = json.dumps(rows[:20], indent=2)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data analyst. Answer questions about tabular data "
                    "concisely and precisely."
                ),
            },
            {
                "role": "user",
                "content": f"Data:\n{data_str[:3000]}\n\nQuestion: {question}",
            },
        ],
    )
    return response["message"]["content"]
```

</details>